# Practical 9 — Data Ingestion, Validation, and Pipelines

**Dataset:** UCI Adult Census Income (`adult.csv`, 32,561 rows, 15 columns)

**Stack:** Elasticsearch 9.2.0, Python `elasticsearch` 9.5.1, pandas 3.1.1

This practical builds a complete ingestion workflow:

1. Download and inspect the raw CSV.
2. Detect missing values and duplicates.
3. Clean, validate, derive fields, and assign stable record IDs.
4. Create an explicit strict mapping.
5. Benchmark single-document indexing against the Bulk API.
6. Quarantine and repair malformed documents.
7. Delete by query, back up with `_reindex`, and restore.
8. Normalize raw input with an ingest pipeline.
9. Validate indexed data with aggregations and the document API.
10. Use the reusable `ingest.py` CLI against valid and invalid files.

The canonical index is **`adult_people`**. Supporting demo indices are recreated
each run, so the notebook is idempotent.

## Task 1 — Download the dataset with Python

`download_data.py` uses `urllib.request` to fetch the UCI Adult dataset and then
creates small CSV/JSON samples plus an intentionally invalid CSV for ingestion
tests. Files already present are reused unless `--force` is supplied.

In [1]:
import json
import subprocess
import sys
import time
from copy import deepcopy
from pathlib import Path

import pandas as pd
from elasticsearch import Elasticsearch, helpers
from IPython.display import display

HERE = Path.cwd()
assert HERE.name == "Practical_9", f"Run this notebook from Practical_9, not {HERE}"

DATA_DIR = HERE / "data"
ARTIFACTS = HERE / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

ES_URL = "http://127.0.0.1:9200"
es = Elasticsearch(ES_URL, request_timeout=60)
info = es.info()
print("Cluster:", info["cluster_name"], "| Elasticsearch:", info["version"]["number"])
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)

SOURCE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
RAW_CSV = DATA_DIR / "adult.csv"
SAMPLE_CSV = DATA_DIR / "adult_sample.csv"
SAMPLE_JSON = DATA_DIR / "adult_sample.json"
INVALID_CSV = DATA_DIR / "adult_invalid.csv"
MAPPING = json.loads((HERE / "mapping.json").read_text(encoding="utf-8"))

RAW_COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income",
]
NUMERIC_COLUMNS = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]

Cluster: my-elasticsearch-cluster | Elasticsearch: 9.2.0
Python: 3.14.0 | pandas: 3.1.1


In [2]:
if not RAW_CSV.exists():
    subprocess.run([sys.executable, str(HERE / "download_data.py")], cwd=HERE, check=True)

for path in [RAW_CSV, SAMPLE_CSV, SAMPLE_JSON, INVALID_CSV]:
    assert path.exists(), f"Missing dataset file: {path}"
    print(f"{path.name:22s} {path.stat().st_size:>10,d} bytes")

raw_line_count = sum(1 for _ in RAW_CSV.open(encoding="utf-8"))
print("Physical CSV lines:", f"{raw_line_count:,}")
print("Source:", SOURCE_URL)

adult.csv               3,974,460 bytes
adult_sample.csv          109,052 bytes
adult_sample.json         438,898 bytes
adult_invalid.csv             822 bytes
Physical CSV lines: 32,562
Source: https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data


## Task 2 — Read and inspect the raw dataset

Read every field as text first. This preserves raw values and makes the cleaning
step explicit instead of silently coercing invalid values to null.

In [3]:
raw = pd.read_csv(RAW_CSV, dtype=str, keep_default_na=False)

print("Shape:", raw.shape)
print("\nColumn dtypes:")
print(raw.dtypes.to_string())
display(raw.head(3))

Shape: (32561, 15)

Column dtypes:
age               str
workclass         str
fnlwgt            str
education         str
education_num     str
marital_status    str
occupation        str
relationship      str
race              str
sex               str
capital_gain      str
capital_loss      str
hours_per_week    str
native_country    str
income            str


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K


## Task 3 — Detect missing values and duplicates

In this dataset missing values are encoded as `?` (sometimes with surrounding
spaces). The official totals are **4,262 missing cells** and **24 exact duplicate
rows**.

In [4]:
missing_counts = {
    column: int((raw[column].str.strip() == "?").sum())
    for column in raw.columns
}
missing_table = (
    pd.Series(missing_counts, name="missing")
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .to_frame()
)
MISSING_TOTAL = int(missing_table["missing"].sum())
DUPLICATE_COUNT = int(raw.duplicated().sum())

display(missing_table)
print("Total missing cells:", MISSING_TOTAL)
print("Exact duplicate rows:", DUPLICATE_COUNT)
print("\nRows touching at least one missing value:", int(raw.apply(
    lambda row: (row.str.strip() == "?").any(), axis=1
).sum()))

assert MISSING_TOTAL == 4262
assert DUPLICATE_COUNT == 24
assert missing_counts["workclass"] == 1836
assert missing_counts["occupation"] == 1843
assert missing_counts["native_country"] == 583

,missing
occupation,1843
workclass,1836
native_country,583


Total missing cells: 4262
Exact duplicate rows: 24



Rows touching at least one missing value: 2399


## Task 4 — Clean, validate, and derive fields

Cleaning rules:

- Trim every string; map `?` to missing.
- Drop exact duplicates.
- Fill missing categorical values with `Unknown`.
- Convert six numeric fields to integers and enforce sensible ranges.
- Lowercase `sex` and `income`.
- Derive `net_capital`, `is_high_income`, `age_band`, and `hours_band`.
- Add `record_id` and `source_row` for traceability back to the CSV.

In [5]:
clean = raw.copy()
for column in clean.columns:
    clean[column] = clean[column].str.strip()

clean = clean.replace("?", pd.NA)
clean = clean.drop_duplicates()

clean["source_row"] = clean.index + 2
clean = clean.reset_index(drop=True)
clean["record_id"] = [f"AD-{n:06d}" for n in range(1, len(clean) + 1)]

for column in ["workclass", "occupation", "native_country"]:
    clean[column] = clean[column].fillna("Unknown")

for column in NUMERIC_COLUMNS:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")

numeric_nulls = clean[NUMERIC_COLUMNS].isna().sum()
print("Numeric fields that failed conversion:")
print(numeric_nulls.to_string())
assert int(numeric_nulls.sum()) == 0

assert clean["age"].between(17, 90).all()
assert clean["hours_per_week"].between(1, 99).all()
assert clean["education_num"].between(1, 16).all()
assert set(clean["sex"]) == {"Male", "Female"}
assert set(clean["income"]) == {"<=50K", ">50K"}

clean["sex"] = clean["sex"].str.lower()
clean["income"] = clean["income"].str.lower()
clean["net_capital"] = clean["capital_gain"] - clean["capital_loss"]
clean["is_high_income"] = clean["income"] == ">50k"

clean["age_band"] = pd.cut(
    clean["age"],
    bins=[16, 24, 34, 44, 54, 64, 100],
    labels=["17-24", "25-34", "35-44", "45-54", "55-64", "65+"],
).astype(str)
clean["hours_band"] = pd.cut(
    clean["hours_per_week"],
    bins=[0, 34, 40, 100],
    labels=["part-time", "full-time", "overtime"],
).astype(str)

CLEAN_COLUMNS = ["record_id", "source_row"] + RAW_COLUMNS + [
    "net_capital", "is_high_income", "age_band", "hours_band",
]
clean = clean[CLEAN_COLUMNS]
CLEAN_DF = clean.copy()
DOCUMENTS = json.loads(clean.to_json(orient="records"))
CLEAN_COUNT = len(clean)

print("Raw rows:", f"{len(raw):,}")
print("Clean rows:", f"{CLEAN_COUNT:,}")
print("Duplicates removed:", f"{len(raw) - CLEAN_COUNT:,}")
print("\nIncome counts:")
print(clean["income"].value_counts().to_string())
print("\nUnknown counts:")
print({c: int((clean[c] == "Unknown").sum()) for c in ["workclass", "occupation", "native_country"]})
print("\nFirst document:")
print(json.dumps(DOCUMENTS[0], indent=2))

assert CLEAN_COUNT == 32537
assert len({doc["record_id"] for doc in DOCUMENTS}) == CLEAN_COUNT

Numeric fields that failed conversion:
age               0
fnlwgt            0
education_num     0
capital_gain      0
capital_loss      0
hours_per_week    0


Raw rows: 32,561
Clean rows: 32,537
Duplicates removed: 24

Income counts:
income
<=50k    24698
>50k      7839

Unknown counts:
{'workclass': 1836, 'occupation': 1843, 'native_country': 582}

First document:
{
  "record_id": "AD-000001",
  "source_row": 2,
  "age": 39,
  "workclass": "State-gov",
  "fnlwgt": 77516,
  "education": "Bachelors",
  "education_num": 13,
  "marital_status": "Never-married",
  "occupation": "Adm-clerical",
  "relationship": "Not-in-family",
  "race": "White",
  "sex": "male",
  "capital_gain": 2174,
  "capital_loss": 0,
  "hours_per_week": 40,
  "native_country": "United-States",
  "income": "<=50k",
  "net_capital": 2174,
  "is_high_income": false,
  "age_band": "35-44",
  "hours_band": "full-time"
}


## Task 5 — Create an explicit strict mapping

`mapping.json` declares all 21 properties and sets `dynamic: strict`. Unknown
fields are rejected, so schema drift cannot silently create new mappings.

In [6]:
if es.indices.exists(index="adult_people"):
    es.indices.delete(index="adult_people")

es.indices.create(index="adult_people", **MAPPING)
mapping_properties = es.indices.get_mapping(index="adult_people")["adult_people"]["mappings"]["properties"]

print("Field types:")
for name in sorted(mapping_properties):
    print(f"  {name:18s} {mapping_properties[name].get('type', 'object')}")
print("\nProperties:", len(mapping_properties))
assert int(es.count(index="adult_people")["count"]) == 0

Field types:
  age                integer
  age_band           keyword
  capital_gain       integer
  capital_loss       integer
  education          keyword
  education_num      integer
  fnlwgt             long
  hours_band         keyword
  hours_per_week     integer
  income             keyword
  is_high_income     boolean
  marital_status     keyword
  native_country     keyword
  net_capital        integer
  occupation         keyword
  race               keyword
  record_id          keyword
  relationship       keyword
  sex                keyword
  source_row         integer
  workclass          keyword

Properties: 21


## Task 6 — Index documents one at a time

Indexing one document per request is simple but dominated by HTTP round trips.
Use a fixed sample of 500 documents so the comparison is reproducible.

In [7]:
def recreate_index(name, mapping):
    if es.indices.exists(index=name):
        es.indices.delete(index=name)
    es.indices.create(index=name, **mapping)

SAMPLE_SIZE = 500
benchmark_docs = DOCUMENTS[:SAMPLE_SIZE]

recreate_index("adult_individual_bench", MAPPING)

individual_start = time.perf_counter()
for doc in benchmark_docs:
    es.index(index="adult_individual_bench", id=doc["record_id"], document=doc)
INDIVIDUAL_SECONDS = time.perf_counter() - individual_start

es.indices.refresh(index="adult_individual_bench")
individual_count = int(es.count(index="adult_individual_bench")["count"])
print(f"Individual indexing: {individual_count:,} docs in {INDIVIDUAL_SECONDS:.2f}s")
print(f"Per-document average: {INDIVIDUAL_SECONDS / individual_count * 1000:.1f} ms")
assert individual_count == SAMPLE_SIZE

Individual indexing: 500 docs in 53.53s
Per-document average: 107.1 ms


## Task 7 — Index with the Bulk API

The Bulk API batches actions into one request. First repeat the 500-document
benchmark, then load all 32,537 cleaned documents into **`adult_people`**.

In [8]:
recreate_index("adult_bulk_bench", MAPPING)

bulk_start = time.perf_counter()
bulk_success, bulk_errors = helpers.bulk(
    es,
    ({"_index": "adult_bulk_bench", "_id": doc["record_id"], "_source": doc} for doc in benchmark_docs),
    chunk_size=250,
)
BULK_SECONDS = time.perf_counter() - bulk_start
es.indices.refresh(index="adult_bulk_bench")
bulk_count = int(es.count(index="adult_bulk_bench")["count"])
assert bulk_success == SAMPLE_SIZE and bulk_errors == []
assert bulk_count == SAMPLE_SIZE

full_start = time.perf_counter()
full_success, full_errors = helpers.bulk(
    es,
    ({"_index": "adult_people", "_id": doc["record_id"], "_source": doc} for doc in DOCUMENTS),
    chunk_size=1000,
)
FULL_INDEX_SECONDS = time.perf_counter() - full_start
es.indices.refresh(index="adult_people")
full_count = int(es.count(index="adult_people")["count"])

print(f"Bulk benchmark: {bulk_success:,} docs in {BULK_SECONDS:.2f}s")
print(f"Full load: {full_success:,} docs in {FULL_INDEX_SECONDS:.2f}s "
      f"({full_success / FULL_INDEX_SECONDS:,.0f} docs/s)")
print("Full-load errors:", full_errors)
assert full_success == CLEAN_COUNT and full_errors == []
assert full_count == CLEAN_COUNT

Bulk benchmark: 500 docs in 0.24s
Full load: 32,537 docs in 4.30s (7,567 docs/s)
Full-load errors: []


## Task 8 — Compare the two strategies

Bulk indexing should be dramatically faster because it amortizes network and
per-request overhead across many documents.

In [9]:
SPEEDUP = INDIVIDUAL_SECONDS / BULK_SECONDS
benchmark = {
    "sample_size": SAMPLE_SIZE,
    "individual_seconds": round(INDIVIDUAL_SECONDS, 3),
    "bulk_seconds": round(BULK_SECONDS, 3),
    "speedup": round(SPEEDUP, 1),
    "full_dataset_seconds": round(FULL_INDEX_SECONDS, 3),
    "full_dataset_docs_per_second": round(full_success / FULL_INDEX_SECONDS, 1),
    "documents": CLEAN_COUNT,
}
(ARTIFACTS / "benchmark.json").write_text(json.dumps(benchmark, indent=2), encoding="utf-8")

print(f"Individual: {INDIVIDUAL_SECONDS:.2f}s")
print(f"Bulk:       {BULK_SECONDS:.2f}s")
print(f"Speedup:    {SPEEDUP:.1f}x faster with bulk")
assert BULK_SECONDS < INDIVIDUAL_SECONDS
assert bulk_count == individual_count == SAMPLE_SIZE
assert full_count == CLEAN_COUNT

Individual: 53.53s
Bulk:       0.24s
Speedup:    226.0x faster with bulk


## Task 9 — Handle malformed documents

`helpers.bulk(..., raise_on_error=False)` keeps going and returns successful
counts plus per-document errors. Malformed documents are written to an artifact,
then repaired and retried.

In [10]:
recreate_index("adult_failed_demo", MAPPING)

good_0 = deepcopy(DOCUMENTS[0])
good_1 = deepcopy(DOCUMENTS[1])
bad_age = deepcopy(DOCUMENTS[2])
bad_age["age"] = "not-a-number"
bad_hours = deepcopy(DOCUMENTS[3])
bad_hours["hours_per_week"] = "many"

failure_batch = [good_0, good_1, bad_age, bad_hours]
failure_success, failure_errors = helpers.bulk(
    es,
    ({"_index": "adult_failed_demo", "_id": doc["record_id"], "_source": doc} for doc in failure_batch),
    chunk_size=2,
    raise_on_error=False,
)
es.indices.refresh(index="adult_failed_demo")
failure_count = int(es.count(index="adult_failed_demo")["count"])

FAILED_RECORDS = []
for item in failure_errors:
    detail = item.get("index", item.get("create", {}))
    error = detail.get("error", {})
    record = {
        "record_id": detail.get("_id"),
        "status": detail.get("status"),
        "error_type": error.get("type"),
        "reason": error.get("reason"),
        "document": next(
            (doc for doc in failure_batch if doc["record_id"] == detail.get("_id")), None
        ),
    }
    FAILED_RECORDS.append(record)

(ARTIFACTS / "failed_records.json").write_text(
    json.dumps(FAILED_RECORDS, indent=2), encoding="utf-8"
)

print("Successful documents:", failure_success)
print("Failed documents:", len(failure_errors))
for record in FAILED_RECORDS:
    print(f"  {record['record_id']}: HTTP {record['status']} — {record['reason']}")

assert failure_success == 2
assert len(failure_errors) == 2
assert failure_count == 2

repaired_age = deepcopy(bad_age)
repaired_age["age"] = 45
repaired_age["age_band"] = "45-54"
repaired_hours = deepcopy(bad_hours)
repaired_hours["hours_per_week"] = 40
repaired_hours["hours_band"] = "full-time"

repair_success, repair_errors = helpers.bulk(
    es,
    ({"_index": "adult_failed_demo", "_id": doc["record_id"], "_source": doc}
     for doc in [repaired_age, repaired_hours]),
    raise_on_error=False,
)
es.indices.refresh(index="adult_failed_demo")
repaired_count = int(es.count(index="adult_failed_demo")["count"])
print("Repaired and reindexed:", repair_success)
print("Repair errors:", repair_errors)
print("Final count:", repaired_count)
assert repair_success == 2 and repair_errors == []
assert repaired_count == 4

Successful documents: 2
Failed documents: 2
  AD-000003: HTTP 400 — [1:47] failed to parse field [age] of type [integer] in document with id 'AD-000003'. Preview of field's value: 'not-a-number'
  AD-000004: HTTP 400 — [1:300] failed to parse field [hours_per_week] of type [integer] in document with id 'AD-000004'. Preview of field's value: 'many'
Repaired and reindexed: 2
Repair errors: []
Final count: 4


## Task 10 — Delete by query (with a backup)

Before deleting, copy **`adult_people`** to **`adult_backup`** using `_reindex`.
Then remove every person younger than 25. The expected count comes from pandas,
not from Elasticsearch.

In [11]:
recreate_index("adult_backup", MAPPING)

backup_start = time.perf_counter()
backup_response = es.reindex(
    source={"index": "adult_people"},
    dest={"index": "adult_backup"},
    wait_for_completion=True,
)
BACKUP_SECONDS = time.perf_counter() - backup_start
es.indices.refresh(index="adult_backup")
backup_count = int(es.count(index="adult_backup")["count"])
assert backup_response["failures"] == []
assert backup_count == CLEAN_COUNT

before_delete = int(es.count(index="adult_people")["count"])
EXPECTED_YOUNG = int((CLEAN_DF["age"] < 25).sum())

delete_response = es.delete_by_query(
    index="adult_people",
    query={"range": {"age": {"lt": 25}}},
    refresh=True,
    conflicts="proceed",
)
after_delete = int(es.count(index="adult_people")["count"])
young_after_delete = int(es.count(
    index="adult_people", query={"range": {"age": {"lt": 25}}}
)["count"])

print(f"Backup: {backup_count:,} docs in {BACKUP_SECONDS:.2f}s")
print(f"Before delete: {before_delete:,}")
print(f"Deleted: {delete_response['deleted']:,} (expected {EXPECTED_YOUNG:,})")
print(f"After delete: {after_delete:,}")
print(f"Documents younger than 25 remaining: {young_after_delete}")

assert delete_response["deleted"] == EXPECTED_YOUNG == 5562
assert after_delete == CLEAN_COUNT - EXPECTED_YOUNG
assert young_after_delete == 0

Backup: 32,537 docs in 1.69s
Before delete: 32,537
Deleted: 5,562 (expected 5,562)
After delete: 26,975
Documents younger than 25 remaining: 0


## Task 11 — Restore the deleted subset with `_reindex`

Copy only the affected documents from the backup back into **`adult_people`**.
Because `_reindex` uses the document IDs, documents are restored in place.

In [12]:
restore_response = es.reindex(
    source={
        "index": "adult_backup",
        "query": {"range": {"age": {"lt": 25}}},
    },
    dest={"index": "adult_people"},
    wait_for_completion=True,
)
es.indices.refresh(index="adult_people")
restored_count = int(es.count(index="adult_people")["count"])
young_restored = int(es.count(
    index="adult_people", query={"range": {"age": {"lt": 25}}}
)["count"])

print("Created:", restore_response["created"])
print("Updated:", restore_response["updated"])
print("Failures:", restore_response["failures"])
print("Total documents:", f"{restored_count:,}")
print("Restored young documents:", f"{young_restored:,}")

assert restore_response["failures"] == []
assert restore_response["created"] == EXPECTED_YOUNG
assert restored_count == CLEAN_COUNT
assert young_restored == EXPECTED_YOUNG

Created: 5562
Updated: 0
Failures: []
Total documents: 32,537
Restored young documents: 5,562


## Task 12 — Create an ingest pipeline

The pipeline normalizes raw CSV values **before** indexing:

- trim every string field;
- lowercase `sex` and `income`;
- replace hyphens in `native_country`;
- convert numeric strings;
- derive capital, income, age-band, and hours-band fields;
- stamp pipeline metadata and remove `temp_note`.

In [13]:
PIPELINE_ID = "adult_normalize_pipeline"
TRIM_FIELDS = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income",
]

normalize_script = (
    "ctx.net_capital = ctx.capital_gain - ctx.capital_loss; "
    "ctx.is_high_income = ctx.income == '>50k'; "
    "def age = ctx.age; "
    "if (age < 25) { ctx.age_band = '17-24'; } "
    "else if (age < 35) { ctx.age_band = '25-34'; } "
    "else if (age < 45) { ctx.age_band = '35-44'; } "
    "else if (age < 55) { ctx.age_band = '45-54'; } "
    "else if (age < 65) { ctx.age_band = '55-64'; } "
    "else { ctx.age_band = '65+'; } "
    "def hours = ctx.hours_per_week; "
    "if (hours < 35) { ctx.hours_band = 'part-time'; } "
    "else if (hours <= 40) { ctx.hours_band = 'full-time'; } "
    "else { ctx.hours_band = 'overtime'; }"
)

processors = [{"trim": {"field": field, "ignore_missing": True}} for field in TRIM_FIELDS]
processors += [
    {"lowercase": {"field": "sex", "ignore_missing": True}},
    {"lowercase": {"field": "income", "ignore_missing": True}},
    {"gsub": {"field": "native_country", "pattern": "-", "replacement": " ", "ignore_missing": True}},
]
for field in ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]:
    processors.append({"convert": {"field": field, "type": "integer"}})
processors.append({"convert": {"field": "fnlwgt", "type": "long"}})
processors += [
    {"script": {"lang": "painless", "source": normalize_script}},
    {"set": {"field": "pipeline_name", "value": PIPELINE_ID}},
    {"set": {"field": "pipeline_version", "value": 1}},
    {"remove": {"field": "temp_note", "ignore_missing": True}},
]

es.ingest.put_pipeline(
    id=PIPELINE_ID,
    description="Normalize raw Adult Census documents before indexing",
    processors=processors,
)

stored_pipeline = es.ingest.get_pipeline(id=PIPELINE_ID)[PIPELINE_ID]
print("Pipeline:", PIPELINE_ID)
print("Processors:", len(stored_pipeline["processors"]))
print("First:", stored_pipeline["processors"][0])
print("Last:", stored_pipeline["processors"][-1])
assert len(stored_pipeline["processors"]) == len(processors)

Pipeline: adult_normalize_pipeline
Processors: 28
First: {'trim': {'field': 'age', 'ignore_missing': True}}
Last: {'remove': {'field': 'temp_note', 'ignore_missing': True}}


## Task 13 — Simulate and use the pipeline

Simulation shows the transformation without indexing. Then 100 raw rows are
sent through the pipeline into **`adult_pipeline_demo`**.

In [14]:
raw_preview = raw.iloc[0].to_dict()
raw_preview["temp_note"] = "remove me"

simulation = es.ingest.simulate(id=PIPELINE_ID, docs=[{"_source": raw_preview}])
assert "error" not in simulation["docs"][0], simulation
normalized_preview = simulation["docs"][0]["doc"]["_source"]

print("Before:")
print({k: raw_preview[k] for k in ["age", "sex", "income", "native_country", "temp_note"]})
print("\nAfter:")
print({k: normalized_preview.get(k) for k in [
    "age", "sex", "income", "native_country", "net_capital",
    "is_high_income", "age_band", "hours_band", "pipeline_name", "temp_note",
]})
assert normalized_preview["sex"] == "male"
assert normalized_preview["income"] == "<=50k"
assert normalized_preview["native_country"] == "United States"
assert normalized_preview["age"] == 39
assert normalized_preview["pipeline_name"] == PIPELINE_ID
assert "temp_note" not in normalized_preview

Before:
{'age': '39', 'sex': ' Male', 'income': ' <=50K', 'native_country': ' United-States', 'temp_note': 'remove me'}

After:
{'age': 39, 'sex': 'male', 'income': '<=50k', 'native_country': 'United States', 'net_capital': 2174, 'is_high_income': False, 'age_band': '35-44', 'hours_band': 'full-time', 'pipeline_name': 'adult_normalize_pipeline', 'temp_note': None}


In [15]:
pipeline_properties = deepcopy(MAPPING["mappings"]["properties"])
pipeline_properties["pipeline_name"] = {"type": "keyword"}
pipeline_properties["pipeline_version"] = {"type": "integer"}
PIPELINE_MAPPING = {
    "settings": {"number_of_shards": 1, "number_of_replicas": 0},
    "mappings": {"properties": pipeline_properties},
}
recreate_index("adult_pipeline_demo", PIPELINE_MAPPING)

raw_batch = json.loads(raw.head(100).to_json(orient="records"))
pipeline_success, pipeline_errors = helpers.bulk(
    es,
    ({"_index": "adult_pipeline_demo", "_source": doc} for doc in raw_batch),
    chunk_size=50,
    pipeline=PIPELINE_ID,
    raise_on_error=False,
)
es.indices.refresh(index="adult_pipeline_demo")
pipeline_count = int(es.count(index="adult_pipeline_demo")["count"])

pipeline_sample = es.search(index="adult_pipeline_demo", size=1)["hits"]["hits"][0]["_source"]
print("Indexed through pipeline:", pipeline_success, "| errors:", len(pipeline_errors))
print("Count:", pipeline_count)
print(json.dumps(pipeline_sample, indent=2))

assert pipeline_success == 100 and pipeline_errors == []
assert pipeline_count == 100
assert pipeline_sample["income"] in {"<=50k", ">50k"}
assert pipeline_sample["sex"] in {"male", "female"}
assert "-" not in pipeline_sample["native_country"]
assert pipeline_sample["pipeline_name"] == PIPELINE_ID

Indexed through pipeline: 100 | errors: 0
Count: 100
{
  "income": "<=50k",
  "pipeline_version": 1,
  "education": "Bachelors",
  "education_num": 13,
  "occupation": "Adm-clerical",
  "age_band": "35-44",
  "race": "White",
  "sex": "male",
  "hours_per_week": 40,
  "pipeline_name": "adult_normalize_pipeline",
  "fnlwgt": 77516,
  "capital_loss": 0,
  "native_country": "United States",
  "marital_status": "Never-married",
  "workclass": "State-gov",
  "is_high_income": false,
  "relationship": "Not-in-family",
  "net_capital": 2174,
  "hours_band": "full-time",
  "age": 39,
  "capital_gain": 2174
}


## Task 14 — Validate the indexed data

Validation compares Elasticsearch aggregations with pandas results and inspects
the stored mapping and documents. The report is written to
`artifacts/validation_report.json`.

In [16]:
income_agg = es.search(
    index="adult_people", size=0,
    aggs={"income": {"terms": {"field": "income", "size": 10}}},
)
income_buckets = {
    bucket["key"]: int(bucket["doc_count"])
    for bucket in income_agg["aggregations"]["income"]["buckets"]
}
expected_income = {str(k): int(v) for k, v in CLEAN_DF["income"].value_counts().items()}
assert income_buckets == expected_income

avg_response = es.search(
    index="adult_people", size=0,
    aggs={"avg_age": {"avg": {"field": "age"}}},
)
avg_age = float(avg_response["aggregations"]["avg_age"]["value"])
expected_avg_age = float(CLEAN_DF["age"].mean())
assert abs(avg_age - expected_avg_age) < 1e-4

unknown_aggs = es.search(
    index="adult_people", size=0,
    aggs={
        field: {"filter": {"term": {field: "Unknown"}}}
        for field in ["workclass", "occupation", "native_country"]
    },
)
unknown_indexed = {
    field: int(unknown_aggs["aggregations"][field]["doc_count"])
    for field in ["workclass", "occupation", "native_country"]
}
unknown_expected = {
    field: int((CLEAN_DF[field] == "Unknown").sum())
    for field in unknown_indexed
}
assert unknown_indexed == unknown_expected

sex_agg = es.search(
    index="adult_people", size=0,
    aggs={"sex": {"terms": {"field": "sex"}}},
)
sex_buckets = {
    bucket["key"]: int(bucket["doc_count"])
    for bucket in sex_agg["aggregations"]["sex"]["buckets"]
}
assert sex_buckets == {str(k): int(v) for k, v in CLEAN_DF["sex"].value_counts().items()}

age_band_agg = es.search(
    index="adult_people", size=0,
    aggs={"age_band": {"terms": {"field": "age_band", "size": 10}}},
)
age_band_buckets = {
    bucket["key"]: int(bucket["doc_count"])
    for bucket in age_band_agg["aggregations"]["age_band"]["buckets"]
}
assert age_band_buckets == {
    str(k): int(v) for k, v in CLEAN_DF["age_band"].value_counts().items()
}

sample_id = DOCUMENTS[0]["record_id"]
sample_doc = es.get(
    index="adult_people",
    id=sample_id,
    source_includes=[
        "record_id", "age", "sex", "income", "net_capital",
        "is_high_income", "age_band", "hours_band",
    ],
)
assert sample_doc["found"] is True
assert sample_doc["_source"]["record_id"] == sample_id
assert "?" not in json.dumps(sample_doc["_source"])

stored_mapping = es.indices.get_mapping(index="adult_people")["adult_people"]["mappings"]
assert stored_mapping["dynamic"] == "strict"
assert stored_mapping["properties"]["age"]["type"] == "integer"
assert stored_mapping["properties"]["sex"]["type"] == "keyword"
assert stored_mapping["properties"]["is_high_income"]["type"] == "boolean"

pipeline_sample = es.search(index="adult_pipeline_demo", size=1)["hits"]["hits"][0]["_source"]
failed_demo_count = int(es.count(index="adult_failed_demo")["count"])
assert failed_demo_count == 4
assert pipeline_sample["pipeline_version"] == 1

VALIDATION_REPORT = {
    "dataset": {
        "source": SOURCE_URL,
        "raw_rows": len(raw),
        "clean_rows": CLEAN_COUNT,
        "duplicates_removed": len(raw) - CLEAN_COUNT,
        "missing_cells": MISSING_TOTAL,
        "missing_by_field": missing_counts,
    },
    "cleaning": {
        "unknown_workclass": int((CLEAN_DF["workclass"] == "Unknown").sum()),
        "unknown_occupation": int((CLEAN_DF["occupation"] == "Unknown").sum()),
        "unknown_native_country": int((CLEAN_DF["native_country"] == "Unknown").sum()),
        "avg_age": round(expected_avg_age, 4),
        "income": expected_income,
        "sex": sex_buckets,
    },
    "mapping": {
        "index": "adult_people",
        "dynamic": stored_mapping["dynamic"],
        "properties": len(stored_mapping["properties"]),
    },
    "indexing": benchmark,
    "failures": {
        "successful": failure_success,
        "failed": len(failure_errors),
        "records": FAILED_RECORDS,
        "repaired_count": repaired_count,
    },
    "delete_and_restore": {
        "backup_count": backup_count,
        "backup_seconds": round(BACKUP_SECONDS, 3),
        "deleted": int(delete_response["deleted"]),
        "expected_deleted": EXPECTED_YOUNG,
        "after_delete": after_delete,
        "restored": int(restore_response["created"]),
        "final_count": restored_count,
    },
    "pipeline": {
        "id": PIPELINE_ID,
        "processors": len(stored_pipeline["processors"]),
        "demo_documents": pipeline_count,
        "sample": pipeline_sample,
    },
    "validation": {
        "income_buckets": income_buckets,
        "avg_age_indexed": round(avg_age, 4),
        "avg_age_pandas": round(expected_avg_age, 4),
        "unknown_fields": unknown_indexed,
        "sample_document": sample_doc["_source"],
        "failed_demo_count": failed_demo_count,
    },
}
(ARTIFACTS / "validation_report.json").write_text(
    json.dumps(VALIDATION_REPORT, indent=2), encoding="utf-8"
)

print("Indexed documents:", f"{full_count:,}")
print("Income buckets:", income_buckets)
print("Average age:", f"{avg_age:.4f} (pandas {expected_avg_age:.4f})")
print("Unknown fields:", unknown_indexed)
print("Sample document:", json.dumps(sample_doc["_source"], indent=2))
print("\nAll validation checks passed.")

Indexed documents: 32,537
Income buckets: {'<=50k': 24698, '>50k': 7839}
Average age: 38.5855 (pandas 38.5855)
Unknown fields: {'workclass': 1836, 'occupation': 1843, 'native_country': 582}
Sample document: {
  "record_id": "AD-000001",
  "age": 39,
  "sex": "male",
  "income": "<=50k",
  "net_capital": 2174,
  "is_high_income": false,
  "age_band": "35-44",
  "hours_band": "full-time"
}

All validation checks passed.


## Advanced assignment — reusable ingestion CLI

`ingest.py` packages the workflow as a command-line tool with:

- CSV, JSON, and JSONL input;
- explicit mapping file, batch size, and optional ingest pipeline;
- deduplication, validation, quarantine files, stats files, and logs;
- `--recreate`, `--limit`, `--keep-duplicates`, and `--dry-run` options.

The notebook now drives the CLI as a subprocess against the full CSV, the JSON
sample, and the intentionally invalid CSV.

In [17]:
def run_ingest(label, args):
    stats_path = ARTIFACTS / f"ingestion_stats_{label}.json"
    reject_path = ARTIFACTS / f"rejected_{label}.jsonl"
    log_path = ARTIFACTS / f"ingestion_{label}.log"
    command = [
        sys.executable,
        str(HERE / "ingest.py"),
        "--index", f"adult_ingest_{label}",
        "--mapping", str(HERE / "mapping.json"),
        "--recreate",
        "--stats-file", str(stats_path),
        "--reject-file", str(reject_path),
        "--log-file", str(log_path),
        *args,
    ]
    started = time.perf_counter()
    completed = subprocess.run(
        command, cwd=HERE, capture_output=True, text=True, timeout=180
    )
    elapsed = time.perf_counter() - started
    print(f"\n--- {label} ---")
    print("Exit code:", completed.returncode, f"| wall time: {elapsed:.2f}s")
    if completed.returncode != 0:
        print(completed.stdout[-1500:])
        print(completed.stderr[-1500:])
        raise RuntimeError(f"ingest.py failed for {label}")
    print("\n".join(completed.stdout.strip().splitlines()[-6:]))
    stats = json.loads(stats_path.read_text(encoding="utf-8"))
    print({key: stats[key] for key in [
        "read", "duplicates_removed", "valid", "invalid", "indexed", "rejected_total"
    ]})
    return stats, reject_path, log_path


full_stats, _, _ = run_ingest("csv", [
    "--input", str(RAW_CSV),
    "--batch-size", "1000",
])
json_stats, _, _ = run_ingest("json", [
    "--input", str(SAMPLE_JSON),
    "--batch-size", "500",
])
invalid_stats, invalid_reject_path, _ = run_ingest("invalid", [
    "--input", str(INVALID_CSV),
    "--batch-size", "2",
])

es.indices.refresh(index="adult_ingest_csv")
es.indices.refresh(index="adult_ingest_json")
es.indices.refresh(index="adult_ingest_invalid")
script_counts = {
    "csv": int(es.count(index="adult_ingest_csv")["count"]),
    "json": int(es.count(index="adult_ingest_json")["count"]),
    "invalid": int(es.count(index="adult_ingest_invalid")["count"]),
}
print("\nCLI index counts:", script_counts)

assert full_stats["read"] == 32561
assert full_stats["duplicates_removed"] == 24
assert full_stats["indexed"] == 32537
assert script_counts["csv"] == 32537
assert json_stats["read"] == 1000 and json_stats["indexed"] == 1000
assert script_counts["json"] == 1000
assert invalid_stats["read"] == 6
assert invalid_stats["invalid"] == 2
assert invalid_stats["indexed"] == 4
assert script_counts["invalid"] == 4
assert len(invalid_reject_path.read_text(encoding="utf-8").strip().splitlines()) == 2

VALIDATION_REPORT["script_ingestion"] = {
    "csv": full_stats,
    "json": json_stats,
    "invalid": invalid_stats,
    "index_counts": script_counts,
}
(ARTIFACTS / "validation_report.json").write_text(
    json.dumps(VALIDATION_REPORT, indent=2), encoding="utf-8"
)
print("Updated:", ARTIFACTS / "validation_report.json")


--- csv ---
Exit code: 0 | wall time: 13.07s
  "elapsed_seconds": 10.462,
  "bulk_seconds": 6.293,
  "docs_per_second": 3110.2,
  "started_at": "2026-11-19T06:46:44.593019+00:00",
  "finished_at": "2026-11-19T06:46:55.054561+00:00"
}
{'read': 32561, 'duplicates_removed': 24, 'valid': 32537, 'invalid': 0, 'indexed': 32537, 'rejected_total': 0}



--- json ---
Exit code: 0 | wall time: 5.26s
  "elapsed_seconds": 2.133,
  "bulk_seconds": 0.207,
  "docs_per_second": 468.9,
  "started_at": "2026-11-19T06:46:58.188556+00:00",
  "finished_at": "2026-11-19T06:47:00.321387+00:00"
}
{'read': 1000, 'duplicates_removed': 0, 'valid': 1000, 'invalid': 0, 'indexed': 1000, 'rejected_total': 0}



--- invalid ---
Exit code: 0 | wall time: 5.07s
  "elapsed_seconds": 2.513,
  "bulk_seconds": 0.196,
  "docs_per_second": 1.6,
  "started_at": "2026-11-19T06:47:02.869604+00:00",
  "finished_at": "2026-11-19T06:47:05.382381+00:00"
}


{'read': 6, 'duplicates_removed': 0, 'valid': 4, 'invalid': 2, 'indexed': 4, 'rejected_total': 2}

CLI index counts: {'csv': 32537, 'json': 1000, 'invalid': 4}
Updated: C:\D-Drive\Elastic Search Practicals\Practical_9\artifacts\validation_report.json


## Summary

- 32,561 raw rows became **32,537 clean documents** after removing 24 duplicates.
- 4,262 missing cells were handled; unknown categories are explicit `Unknown`.
- Bulk indexing replaced per-document requests and is dozens to hundreds of
  times faster on the benchmark sample.
- Malformed documents are quarantined in `artifacts/failed_records.json`, then
  repaired and retried.
- `_reindex` provided a backup and restored the 5,562 deleted documents exactly.
- The ingest pipeline normalizes raw strings and derives fields before indexing.
- `ingest.py` reproduces the workflow from the command line and writes stats,
  logs, and rejected-record files.

**Main indices:** `adult_people`, `adult_backup`, `adult_pipeline_demo`,
`adult_ingest_csv`, `adult_ingest_json`, `adult_ingest_invalid`.

**Artifacts:** `benchmark.json`, `failed_records.json`,
`validation_report.json`, ingestion stats, logs, and rejected-record files.